In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import ImageFilter
import os
import numpy as np


#1 Download base CIFAR-10 image dataset
raw_cifer = datasets.CIFAR10(root='./data', train=True, download=True)
print("Base dataset downloaded")



In [ ]:
from torch._higher_order_ops.invoke_subgraph import trace_joint_graph
#2 Define custom dataset that creates Clean(0) vs corrupted (1) image classes

class CorruptedImageDataset(Dataset):
  def __init__(self, raw_dataset):
    self.raw_dataset = raw_dataset
    self.transform = transforms.Compose ([
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229,0.224,0.225])
    ])

  def __len__(self):
    return len(self.raw_dataset)

  def __getitem__(self, idx):
    img, _ = self.raw_dataset[idx]

    #Assign 50% of images as clean(0) and 50% as corrputed (1)
    is_corrupted = 1 if idx % 2 == 1 else 0

    if is_corrupted:
      #Apply severe Gaussian blur and artificial noise
      img = img.filter(ImageFilter.GaussianBlur(radius=3))
      img_array = np.array(img).astype(np.float32)
      noise = np.random.normal(0,25,img_array.shape)
      img_array = np.clip(img_array + noise, 0, 255).astype(np.uint8)
      img = transforms.functional.to_pil_image(img_array)

    tensor_img = self.transform(img)
    return tensor_img, is_corrupted

#3 Create dataset and perform 80/20 train/test split

print("Generating clean vs corrupted image dataset")

full_dataset = CorruptedImageDataset (raw_cifer)
train_size = int(0.8*len(full_dataset))
test_size = len(full_dataset)- train_size
train_dataset, test_dataset = random_split (full_dataset, [train_size, test_size])
print("Dataset generated")

#4 Create PyTorch DataLoaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"\n---SUCCESS ---")
print(f"Total Images Prepared: {len(full_dataset)}")
print(f"Training Batches: {len(train_loader)} | Testing Batches: {len(test_loader)}")
print(f"Target Classes: 0 = Clean Image, 1 = Corrupted Image")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class ImageCorruptionClassifier(nn.Module):
    def __init__(self):
        super(ImageCorruptionClassifier, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 32 * 32, 64)
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageCorruptionClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"Model loaded onto: {device}")

In [ ]:
# Train for 3 epochs
epochs = 3
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f} - Accuracy: {(correct/total)*100:.2f}%")

# Test model accuracy
model.eval()
test_correct, test_total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

print(f"\nFinal Test Accuracy: {(test_correct/test_total)*100:.2f}%")

In [17]:
# 1. Save the model weights locally in Colab
torch.save(model.state_dict(), 'image_corruption_resnet.pth')
print("Model weights successfully saved as 'image_corruption_resnet.pth'")

# 2. Download the saved model weights file directly to your computer
from google.colab import files
files.download('image_corruption_resnet.pth')

Model weights successfully saved as 'image_corruption_resnet.pth'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>